In [1]:
from pathlib import Path
import json
import pandas as pd

cwd = Path.cwd()

PROJECT_ROOT = next(
    (
        p for p in [cwd, *cwd.parents]
        if (p / "data" / "raw" / "ai4i2020.csv").exists()
    ),
    None
)

if PROJECT_ROOT is None:
    raise FileNotFoundError("Project root not found.")

METRICS_DIR = PROJECT_ROOT / "artifacts" / "metrics"
REPORTS_DIR = PROJECT_ROOT / "artifacts" / "reports"

test_metrics_path = METRICS_DIR / "final_test_metrics.csv"
threshold_path = REPORTS_DIR / "threshold_decision.json"

test_metrics = pd.read_csv(test_metrics_path)

with open(threshold_path, "r") as f:
    threshold_decision = json.load(f)

display(test_metrics.T)
print("\nThreshold decision:")
print(json.dumps(threshold_decision, indent=4))

,0
model,Hist_Gradient_Boosting
decision_threshold,0.25
accuracy,0.957
average_precision,0.824608
roc_auc,0.974425
precision,0.433824
recall,0.867647
f1,0.578431
balanced_accuracy,0.913896
true_positives,59



Threshold decision:
{
    "selected_model": "Hist_Gradient_Boosting",
    "selected_threshold": 0.25,
    "selection_data": "Training out-of-fold predictions",
    "minimum_recall_constraint": 0.85,
    "selection_rule": "Maximize precision among tested thresholds with recall >= 0.85",
    "imbalance_strategy": "Class weighting",
    "smote_used": false,
    "random_oversampling_used": false,
    "random_undersampling_used": false,
    "test_set_used_for_threshold_selection": false,
    "oof_precision": 0.44550669216061184,
    "oof_recall": 0.8597785977859779,
    "oof_f1": 0.5869017632241813,
    "oof_true_positives": 233,
    "oof_false_positives": 290,
    "oof_false_negatives": 38
}


In [2]:
m = test_metrics.iloc[0]

print("FINAL TEST-SET INTERPRETATION")
print("=" * 45)

print(f"Model: {m['model']}")
print(f"Threshold: {m['decision_threshold']:.2f}")
print(f"Accuracy: {m['accuracy']:.4f}")
print(f"Average Precision: {m['average_precision']:.4f}")
print(f"ROC-AUC: {m['roc_auc']:.4f}")
print(f"Precision: {m['precision']:.4f}")
print(f"Recall: {m['recall']:.4f}")
print(f"F1-score: {m['f1']:.4f}")
print(f"Balanced Accuracy: {m['balanced_accuracy']:.4f}")

print("\nConfusion-matrix counts")
print(f"True positives: {int(m['true_positives'])}")
print(f"False positives: {int(m['false_positives'])}")
print(f"True negatives: {int(m['true_negatives'])}")
print(f"False negatives: {int(m['false_negatives'])}")

FINAL TEST-SET INTERPRETATION
Model: Hist_Gradient_Boosting
Threshold: 0.25
Accuracy: 0.9570
Average Precision: 0.8246
ROC-AUC: 0.9744
Precision: 0.4338
Recall: 0.8676
F1-score: 0.5784
Balanced Accuracy: 0.9139

Confusion-matrix counts
True positives: 59
False positives: 77
True negatives: 1855
False negatives: 9


In [3]:
report = f"""
# Final Model Analysis

## Selected Configuration

- Model: {m['model']}
- Classification threshold: {m['decision_threshold']:.2f}
- Imbalance strategy: class weighting
- SMOTE: not used
- Random oversampling: not used
- Random undersampling: not used

## Test-Set Results

| Metric | Value |
|---|---:|
| Accuracy | {m['accuracy']:.4f} |
| Average Precision | {m['average_precision']:.4f} |
| ROC-AUC | {m['roc_auc']:.4f} |
| Precision | {m['precision']:.4f} |
| Recall | {m['recall']:.4f} |
| F1-score | {m['f1']:.4f} |
| Balanced Accuracy | {m['balanced_accuracy']:.4f} |

## Confusion Matrix Counts

- True positives: {int(m['true_positives'])}
- False positives: {int(m['false_positives'])}
- True negatives: {int(m['true_negatives'])}
- False negatives: {int(m['false_negatives'])}

## Interpretation

The model detected {int(m['true_positives'])} of the actual failures
in the held-out test set and missed {int(m['false_negatives'])}.
It generated {int(m['false_positives'])} false-positive alerts.

The model demonstrates promising failure-ranking performance, but the
precision indicates that false alarms remain an important concern.

## Limitations

1. The dataset is synthetic and may not represent all real industrial machines.
2. The task detects failure at the current observation; it does not predict remaining useful life or future failure lead time.
3. The selected threshold reflects a provisional recall constraint, not a validated industrial cost function.
4. The test set contains a limited number of failures, so estimates have sampling uncertainty.
5. Real deployment requires validation on operational data, monitoring, and periodic model evaluation.
6. Feature importance and operational explanations should be investigated before relying on predictions for maintenance decisions.

## Conclusion

Hist Gradient Boosting was selected based on out-of-fold average precision
and threshold experiments. A threshold of 0.25 was chosen using training
out-of-fold predictions under a minimum recall criterion of 85%.

The held-out test results are reported separately and were not used to
retune the threshold.
"""

report_path = REPORTS_DIR / "final_model_analysis.md"
report_path.write_text(report.strip(), encoding="utf-8")

print("Final report saved to:", report_path)

Final report saved to: d:\robotic-predictive-maintenance\artifacts\reports\final_model_analysis.md


In [4]:
required_artifacts = [
    PROJECT_ROOT / "artifacts" / "models" / "preprocessor.joblib",
    PROJECT_ROOT / "artifacts" / "metrics" / "baseline_cv_results.csv",
    PROJECT_ROOT / "artifacts" / "metrics" / "oof_model_comparison.csv",
    PROJECT_ROOT / "artifacts" / "metrics" / "threshold_analysis.csv",
    PROJECT_ROOT / "artifacts" / "metrics" / "final_test_metrics.csv",
    PROJECT_ROOT / "artifacts" / "metrics" / "final_test_predictions.csv",
    PROJECT_ROOT / "artifacts" / "reports" / "threshold_decision.json",
    PROJECT_ROOT / "artifacts" / "reports" / "final_evaluation_summary.json",
    PROJECT_ROOT / "artifacts" / "reports" / "final_model_analysis.md",
]

artifact_audit = pd.DataFrame({
    "artifact": [str(p.relative_to(PROJECT_ROOT)) for p in required_artifacts],
    "exists": [p.exists() for p in required_artifacts]
})

display(artifact_audit)

if not artifact_audit["exists"].all():
    print("Some artifacts are missing. Review the audit table.")
else:
    print("All listed artifacts are present.")

,artifact,exists
0,artifacts\models\preprocessor.joblib,True
1,artifacts\metrics\baseline_cv_results.csv,True
2,artifacts\metrics\oof_model_comparison.csv,True
3,artifacts\metrics\threshold_analysis.csv,True
4,artifacts\metrics\final_test_metrics.csv,True
5,artifacts\metrics\final_test_predictions.csv,True
6,artifacts\reports\threshold_decision.json,True
7,artifacts\reports\final_evaluation_summary.json,True
8,artifacts\reports\final_model_analysis.md,True


All listed artifacts are present.
